# RL-APM: Exploratory Data Analysis (EDA)
## Reinforcement Learning for Portfolio Optimization in Equity Markets

This notebook provides diagnostic inspection, quality verification, and visualization of the authentic market dataset used by the RL portfolio agent:
- **Target Tickers:** AAPL, MSFT, GOOGL, AMZN, NVDA
- **Time Horizon:** 2015-01-01 through 2024-12-31 (10-Year Daily Observation Period)
- **Diagnostic Checks:**
  1. Dataset Shape & Date Range
  2. Missing Values & Duplicate Dates Verification
  3. Descriptive Statistics
  4. Historical Closing Prices & Normalized Growth
  5. Daily Percentage Returns & Volatility Profiles
  6. Technical Indicators (SMA, RSI, MACD, Bollinger Bands)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set chart aesthetic defaults
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Resolve relative project directory paths using pathlib
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
print(f"Project Root:   {PROJECT_ROOT}")
print(f"Raw Data Dir:   {RAW_DIR}")
print(f"Processed Dir:  {PROCESSED_DIR}")

### 1. Dataset Shape and Date Range Inspection
Load both raw and processed datasets directly from the CSV files generated by the pipeline.

In [ ]:
summary_records = []
raw_dfs = {}
processed_dfs = {}

for ticker in TICKERS:
    raw_path = RAW_DIR / f"{ticker}.csv"
    proc_path = PROCESSED_DIR / f"{ticker}_processed.csv"
    
    df_raw = pd.read_csv(raw_path, parse_dates=["Date"], index_col="Date") if raw_path.exists() else None
    df_proc = pd.read_csv(proc_path, parse_dates=["Date"], index_col="Date") if proc_path.exists() else None
    
    if df_raw is not None:
        raw_dfs[ticker] = df_raw
    if df_proc is not None:
        processed_dfs[ticker] = df_proc
        
    summary_records.append({
        "Ticker": ticker,
        "Raw Rows": len(df_raw) if df_raw is not None else 0,
        "Raw Columns": df_raw.shape[1] if df_raw is not None else 0,
        "Processed Rows": len(df_proc) if df_proc is not None else 0,
        "Processed Columns": df_proc.shape[1] if df_proc is not None else 0,
        "Start Date": df_proc.index.min().strftime('%Y-%m-%d') if df_proc is not None else "N/A",
        "End Date": df_proc.index.max().strftime('%Y-%m-%d') if df_proc is not None else "N/A",
    })

df_summary = pd.DataFrame(summary_records)
display(df_summary)

### 2. Missing Values and Duplicate Dates Verification
Verify data cleanliness to guarantee clean state inputs for the RL environment.

In [ ]:
hygiene_records = []
for ticker in TICKERS:
    df_raw = raw_dfs.get(ticker)
    df_proc = processed_dfs.get(ticker)
    
    hygiene_records.append({
        "Ticker": ticker,
        "Raw Missing Values": int(df_raw.isna().sum().sum()) if df_raw is not None else -1,
        "Raw Duplicate Dates": int(df_raw.index.duplicated().sum()) if df_raw is not None else -1,
        "Processed Missing Values": int(df_proc.isna().sum().sum()) if df_proc is not None else -1,
        "Processed Duplicate Dates": int(df_proc.index.duplicated().sum()) if df_proc is not None else -1,
    })

df_hygiene = pd.DataFrame(hygiene_records)
display(df_hygiene)

### 3. Descriptive Summary Statistics
Examine price, volume, and indicator distributions across processed features.

In [ ]:
sample_ticker = "AAPL"
if sample_ticker in processed_dfs:
    print(f"=== Descriptive Statistics for {sample_ticker} (Processed Dataset) ===")
    display(processed_dfs[sample_ticker].describe().T)

### 4. Closing-Price Trends
Visualize absolute Adjusted Close trajectories and normalized cumulative asset growth (Base = 100).

In [ ]:
if processed_dfs:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # Panel 1: Adjusted Close prices
    for ticker, df in processed_dfs.items():
        ax1.plot(df.index, df["Adj Close"], label=ticker, linewidth=1.5)
    ax1.set_title("Historical Adjusted Closing Prices (2015 - 2024)", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Price (USD)")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Normalized Growth comparison (Base = 100)
    for ticker, df in processed_dfs.items():
        normalized_prices = (df["Adj Close"] / df["Adj Close"].iloc[0]) * 100
        ax2.plot(df.index, normalized_prices, label=ticker, linewidth=1.5)
    ax2.set_title("Normalized Asset Growth Comparison (Base = 100 at 2015-03-17)", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Normalized Value")
    ax2.set_xlabel("Date")
    ax2.legend(loc="upper left")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 5. Daily Returns and Distribution Profiles
Inspect return volatility clustering and empirical return distribution densities.

In [ ]:
if processed_dfs:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Left: Volatility time series for AAPL and NVDA
    for ticker in ["AAPL", "NVDA"]:
        if ticker in processed_dfs:
            axes[0].plot(processed_dfs[ticker].index, processed_dfs[ticker]["Daily_Return"], label=ticker, alpha=0.6, linewidth=0.8)
    axes[0].set_title("Daily Return Series (Volatility Clustering)", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("Daily Return")
    axes[0].set_xlabel("Date")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Right: Empirical return distributions
    for ticker in TICKERS:
        if ticker in processed_dfs:
            axes[1].hist(processed_dfs[ticker]["Daily_Return"], bins=60, alpha=0.35, label=ticker, density=True)
    axes[1].set_title("Daily Return Distributions (Empirical Densities)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Daily Return")
    axes[1].set_ylabel("Density")
    axes[1].set_xlim(-0.12, 0.12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 6. Technical Indicators Inspection
Examine SMA 20/50, Bollinger Bands, RSI, and MACD for a target asset.

In [ ]:
target_asset = "AAPL"
if target_asset in processed_dfs:
    df_asset = processed_dfs[target_asset].loc["2023-01-01":]
    
    fig, (ax_price, ax_rsi, ax_macd) = plt.subplots(
        3, 1, figsize=(14, 12), sharex=True,
        gridspec_kw={'height_ratios': [2.5, 1, 1.2]}
    )
    
    # Panel 1: Price + SMA 20/50 + Bollinger Bands
    ax_price.plot(df_asset.index, df_asset["Close"], label="Close", color="black", linewidth=1.5)
    ax_price.plot(df_asset.index, df_asset["SMA_20"], label="SMA 20", color="#1f77b4", linestyle="--", alpha=0.8)
    ax_price.plot(df_asset.index, df_asset["SMA_50"], label="SMA 50", color="#ff7f0e", linestyle="--", alpha=0.8)
    ax_price.plot(df_asset.index, df_asset["BB_High"], label="BB Upper", color="#2ca02c", alpha=0.6, linewidth=1)
    ax_price.plot(df_asset.index, df_asset["BB_Low"], label="BB Lower", color="#2ca02c", alpha=0.6, linewidth=1)
    ax_price.fill_between(df_asset.index, df_asset["BB_Low"], df_asset["BB_High"], color="#2ca02c", alpha=0.08, label="BB Band")
    ax_price.set_title(f"{target_asset} Technical Indicators (2023-2024)", fontsize=13, fontweight="bold")
    ax_price.set_ylabel("Price (USD)")
    ax_price.legend(loc="upper left", ncol=3)
    ax_price.grid(True, alpha=0.3)
    
    # Panel 2: RSI 14 with Overbought (70) and Oversold (30) levels
    ax_rsi.plot(df_asset.index, df_asset["RSI_14"], label="RSI 14", color="#9467bd", linewidth=1.5)
    ax_rsi.axhline(70, color="red", linestyle=":", alpha=0.7, label="Overbought (70)")
    ax_rsi.axhline(30, color="green", linestyle=":", alpha=0.7, label="Oversold (30)")
    ax_rsi.set_ylim(15, 85)
    ax_rsi.set_ylabel("RSI")
    ax_rsi.legend(loc="upper left", ncol=3)
    ax_rsi.grid(True, alpha=0.3)
    
    # Panel 3: MACD Line, Signal Line, and Histogram
    ax_macd.plot(df_asset.index, df_asset["MACD"], label="MACD", color="#1f77b4", linewidth=1.5)
    ax_macd.plot(df_asset.index, df_asset["MACD_Signal"], label="Signal", color="#d62728", linestyle="--", linewidth=1.2)
    bar_colors = ["#2ca02c" if val >= 0 else "#d62728" for val in df_asset["MACD_Hist"]]
    ax_macd.bar(df_asset.index, df_asset["MACD_Hist"], label="Histogram", color=bar_colors, alpha=0.5, width=1.2)
    ax_macd.axhline(0, color="black", linewidth=0.8, alpha=0.5)
    ax_macd.set_ylabel("MACD")
    ax_macd.set_xlabel("Date")
    ax_macd.legend(loc="upper left", ncol=4)
    ax_macd.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()